In [ ]:
%pip install mtcnn

In [ ]:
import os
import cv2
from mtcnn.mtcnn import MTCNN
import shutil
from tqdm import tqdm

# Source directories (from the original dataset)
source_fake_dir = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images/train/FAKE"
source_real_dir = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images/train/REAL"

# Destination directories (a new, clean folder in your workspace)
dest_fake_dir = "/kaggle/working/cifake_filtered/train/FAKE"
dest_real_dir = "/kaggle/working/cifake_filtered/train/REAL"

os.makedirs(dest_fake_dir, exist_ok=True)
os.makedirs(dest_real_dir, exist_ok=True)

detector = MTCNN()

def filter_dataset_for_faces(source_dir, dest_dir):
    print(f"Filtering images in: {source_dir}")
    # ... (rest of the filtering function from the previous answer) ...
    image_files = [f for f in os.listdir(source_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    faces_found = 0
    for filename in tqdm(image_files):
        source_path = os.path.join(source_dir, filename)
        dest_path = os.path.join(dest_dir, filename)
        try:
            image = cv2.imread(source_path)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = detector.detect_faces(image_rgb)
            if results:
                shutil.copy(source_path, dest_path)
                faces_found += 1
        except Exception:
            continue
    print(f"Found {faces_found} face images.")

print("--- Filtering FAKE Images from CIFAKE ---")
filter_dataset_for_faces(source_fake_dir, dest_fake_dir)

print("\n--- Filtering REAL Images from CIFAKE ---")
filter_dataset_for_faces(source_real_dir, dest_real_dir)

print("\nCIFAKE filtering complete!")

In [ ]:
import os
import shutil
import glob
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import numpy as np
import torch

print("=" * 70)
print(" OPTIMIZED AI FACE DETECTOR TRAINING ".center(70, "="))
print("=" * 70)

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ Seeds set for reproducibility")

print("\nStarting the dataset merging and splitting process...")

# --- 1. DEFINE SOURCE PATHS ---
real_face_paths = [
    "/kaggle/working/cifake_filtered/train/REAL",
    "/kaggle/input/flickrfaceshq-dataset-ffhq/",
    "/kaggle/input/utkface-new/UTKFace/",
    "/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba/",
    "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real/"
]
ai_face_paths = [
    "/kaggle/working/cifake_filtered/train/FAKE",
    "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake/"
]

# --- 2. DEFINE DESTINATION PATHS ---
base_dir = "/kaggle/working/dataset"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

train_real_dir = os.path.join(train_dir, "real")
train_ai_dir = os.path.join(train_dir, "ai_generated")
val_real_dir = os.path.join(val_dir, "real")
val_ai_dir = os.path.join(val_dir, "ai_generated")

# --- 3. CLEAN UP PREVIOUS RUNS ---
print(f"\nChecking for and removing old directory: {base_dir}")
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
    print("✓ Old directory removed successfully")

# --- 4. CREATE UNIFIED DIRECTORIES ---
print("Creating new, clean directory structure...")
for dir_path in [train_real_dir, train_ai_dir, val_real_dir, val_ai_dir]:
    os.makedirs(dir_path, exist_ok=True)
print("✓ Directory structure created")

# --- 5. FUNCTION TO COLLECT, SPLIT, AND COPY FILES ---
def process_and_copy_files(source_paths, dest_train, dest_val, val_split=0.15, file_limit=None):
    """Enhanced file processing with better validation split"""
    all_files = []
    for path in source_paths:
        if not os.path.exists(path): 
            continue
        all_files.extend(glob.glob(os.path.join(path, '**', '*.jpg'), recursive=True))
        all_files.extend(glob.glob(os.path.join(path, '**', '*.png'), recursive=True))
        all_files.extend(glob.glob(os.path.join(path, '**', '*.jpeg'), recursive=True))
    
    if file_limit and len(all_files) > file_limit:
        all_files = random.sample(all_files, file_limit)
    
    # Better split - 85/15 for more training data
    train_files, val_files = train_test_split(
        all_files, 
        test_size=val_split, 
        random_state=42, 
        shuffle=True
    )
    
    print(f"  Copying {len(train_files):,} training files...")
    for f in tqdm(train_files, desc="  Train"): 
        shutil.copy(f, dest_train)
    
    print(f"  Copying {len(val_files):,} validation files...")
    for f in tqdm(val_files, desc="  Val"): 
        shutil.copy(f, dest_val)
    
    return len(train_files), len(val_files)

# --- 6. EXECUTE THE PROCESS ---
FILE_LIMIT_PER_CLASS = 50000  # Increased from 40k for more data

print("\n" + "=" * 70)
print(" PROCESSING REAL FACES ".center(70, "="))
print("=" * 70)
real_train, real_val = process_and_copy_files(
    real_face_paths, train_real_dir, val_real_dir, 
    file_limit=FILE_LIMIT_PER_CLASS
)

print("\n" + "=" * 70)
print(" PROCESSING AI-GENERATED FACES ".center(70, "="))
print("=" * 70)
ai_train, ai_val = process_and_copy_files(
    ai_face_paths, train_ai_dir, val_ai_dir, 
    file_limit=FILE_LIMIT_PER_CLASS
)

# --- 7. FINAL VERIFICATION ---
print("\n" + "=" * 70)
print(" DATASET SUMMARY ".center(70, "="))
print("=" * 70)
print(f"Training Set:")
print(f"  Real Images:        {len(os.listdir(train_real_dir)):,}")
print(f"  AI-Generated:       {len(os.listdir(train_ai_dir)):,}")
print(f"  Total:              {len(os.listdir(train_real_dir)) + len(os.listdir(train_ai_dir)):,}")
print(f"\nValidation Set:")
print(f"  Real Images:        {len(os.listdir(val_real_dir)):,}")
print(f"  AI-Generated:       {len(os.listdir(val_ai_dir)):,}")
print(f"  Total:              {len(os.listdir(val_real_dir)) + len(os.listdir(val_ai_dir)):,}")
print("=" * 70)

# Train

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

print("\n" + "=" * 70)
print(" ADVANCED MODEL TRAINING ".center(70, "="))
print("=" * 70)

# --- 1. CONFIGURATION ---
train_dir = '/kaggle/working/dataset/train'
validation_dir = '/kaggle/working/dataset/validation'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

# OPTIMIZED HYPERPARAMETERS
IMG_SIZE = 384  # Increased from 260 for better feature extraction
BATCH_SIZE = 16  # Reduced for larger images, better gradients
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_FINETUNE = 1e-7  # More conservative for fine-tuning
EPOCHS_HEAD = 10  # Increased from 5
EPOCHS_FINETUNE = 40  # Increased from 25
WEIGHT_DECAY = 1e-4  # L2 regularization
GRADIENT_CLIP = 0.5  # Prevent exploding gradients

print(f"✓ Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"✓ Batch Size: {BATCH_SIZE}")
print(f"✓ Total Epochs: {EPOCHS_HEAD + EPOCHS_FINETUNE}")

# --- 2. ADVANCED DATA AUGMENTATION ---
print("\n✓ Setting up advanced data augmentation...")

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),  # Reduced from 20 for stability
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Random translation
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.1),  # Occasional grayscale
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),  # Random blur
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1)  # Random erasing for robustness
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(validation_dir, transform=val_transforms)

# Optimized DataLoader settings
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4,  # Increased for faster loading
    pin_memory=True,  # Faster data transfer to GPU
    persistent_workers=True  # Keep workers alive
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE * 2,  # Larger batch for validation
    shuffle=False, 
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print(f"✓ Training samples: {len(train_dataset):,}")
print(f"✓ Validation samples: {len(val_dataset):,}")

# --- 3. MODEL WITH BETTER ARCHITECTURE ---
print("\n✓ Building EfficientNet-B3 model (upgraded from B2)...")

# Using B3 for better performance (more parameters, better accuracy)
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=1)
model = model.to(device)

# Enhanced loss function with label smoothing
class LabelSmoothingBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        target = target * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(pred, target)

criterion = LabelSmoothingBCEWithLogitsLoss(smoothing=0.05)

# Mixed precision training for faster computation
scaler = GradScaler()

# --- 4. TRACKING METRICS ---
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'lr': []
}

def calculate_accuracy(outputs, labels):
    """Calculate binary accuracy"""
    preds = (torch.sigmoid(outputs) > 0.5).float()
    correct = (preds == labels).sum().item()
    return correct / labels.size(0)

def validate(model, val_loader, criterion, device):
    """Validation function"""
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float().unsqueeze(1)
            
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            val_acc += calculate_accuracy(outputs, labels)
    
    return val_loss / len(val_loader), val_acc / len(val_loader)

# --- 5. STAGE 1: TRAIN CLASSIFIER HEAD ---
print("\n" + "=" * 70)
print(" STAGE 1: TRAINING CLASSIFIER HEAD ".center(70, "="))
print("=" * 70)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(
    model.classifier.parameters(), 
    lr=LEARNING_RATE_HEAD,
    weight_decay=WEIGHT_DECAY
)

scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1)

best_val_acc = 0.0

for epoch in range(EPOCHS_HEAD):
    model.train()
    train_loss = 0.0
    train_acc = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_HEAD}")
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().unsqueeze(1)
        
        optimizer.zero_grad(set_to_none=True)  # More efficient
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        train_acc += calculate_accuracy(outputs, labels)
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{calculate_accuracy(outputs, labels):.4f}'
        })
    
    scheduler.step()
    
    # Validation
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_head_model.pth')
        print(f"  ✓ New best model saved! (Val Acc: {val_acc*100:.2f}%)")

# --- 6. STAGE 2: FINE-TUNING WITH DISCRIMINATIVE LEARNING RATES ---
print("\n" + "=" * 70)
print(" STAGE 2: FINE-TUNING ENTIRE MODEL ".center(70, "="))
print("=" * 70)

# Load best head model
model.load_state_dict(torch.load('best_head_model.pth'))

# Unfreeze all parameters
for param in model.parameters():
    param.requires_grad = True

# Discriminative learning rates (lower LR for early layers)
optimizer = optim.AdamW([
    {'params': model.blocks[:4].parameters(), 'lr': LEARNING_RATE_FINETUNE * 0.01},
    {'params': model.blocks[4:].parameters(), 'lr': LEARNING_RATE_FINETUNE * 0.1},
    {'params': model.classifier.parameters(), 'lr': LEARNING_RATE_FINETUNE * 0.5}  # Also lower
], weight_decay=WEIGHT_DECAY)

scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

best_val_acc = 0.0
patience_counter = 0
PATIENCE = 7  # Early stopping patience

for epoch in range(EPOCHS_FINETUNE):
    model.train()
    train_loss = 0.0
    train_acc = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {EPOCHS_HEAD + epoch+1}/{EPOCHS_HEAD + EPOCHS_FINETUNE}")
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().unsqueeze(1)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        train_acc += calculate_accuracy(outputs, labels)
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{calculate_accuracy(outputs, labels):.4f}'
        })
    
    # Validation
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    
    scheduler.step(val_acc)
    
    print(f"\nEpoch {EPOCHS_HEAD + epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': EPOCHS_HEAD + epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'history': history
        }, 'best_model_checkpoint.pth')
        print(f"  ✓ New best model saved! (Val Acc: {val_acc*100:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⚠ Early stopping triggered after {epoch+1} epochs")
        break

# --- 7. SAVE FINAL MODEL ---
print("\n" + "=" * 70)
print(" SAVING FINAL MODEL ".center(70, "="))
print("=" * 70)

# Load best model
checkpoint = torch.load('best_model_checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Save final model
model_filename = 'ai_face_detector_efficientnetb3_optimized.pth'
torch.save(model.state_dict(), model_filename)
print(f"✓ Final model saved as: {model_filename}")
print(f"✓ Best validation accuracy: {checkpoint['val_acc']*100:.2f}%")

# --- 8. PLOT TRAINING HISTORY ---
print("\n✓ Generating training curves...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(x=EPOCHS_HEAD, color='r', linestyle='--', label='Fine-tuning starts')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot([a*100 for a in history['train_acc']], label='Train Acc', linewidth=2)
axes[1].plot([a*100 for a in history['val_acc']], label='Val Acc', linewidth=2)
axes[1].axvline(x=EPOCHS_HEAD, color='r', linestyle='--', label='Fine-tuning starts')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate curve
axes[2].plot(history['lr'], linewidth=2, color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
print("✓ Training curves saved as: training_history.png")

print("\n" + "=" * 70)
print(" TRAINING COMPLETE! ".center(70, "="))
print("=" * 70)

# Test

In [ ]:
import os
import shutil
import glob
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import numpy as np
import torch

print("=" * 70)
print(" OPTIMIZED AI FACE DETECTOR TRAINING ".center(70, "="))
print("=" * 70)

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ Seeds set for reproducibility")

print("\nStarting the dataset merging and splitting process...")

# --- 1. DEFINE SOURCE PATHS ---
real_face_paths = [
    "/kaggle/working/cifake_filtered/train/REAL",
    "/kaggle/input/flickrfaceshq-dataset-ffhq/",
    "/kaggle/input/utkface-new/UTKFace/",
    "/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba/",
    "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real/"
]
ai_face_paths = [
    "/kaggle/working/cifake_filtered/train/FAKE",
    "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake/"
]

# --- 2. DEFINE DESTINATION PATHS ---
base_dir = "/kaggle/working/dataset"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "validation")

train_real_dir = os.path.join(train_dir, "real")
train_ai_dir = os.path.join(train_dir, "ai_generated")
val_real_dir = os.path.join(val_dir, "real")
val_ai_dir = os.path.join(val_dir, "ai_generated")

# --- 3. CLEAN UP PREVIOUS RUNS ---
print(f"\nChecking for and removing old directory: {base_dir}")
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
    print("✓ Old directory removed successfully")

# --- 4. CREATE UNIFIED DIRECTORIES ---
print("Creating new, clean directory structure...")
for dir_path in [train_real_dir, train_ai_dir, val_real_dir, val_ai_dir]:
    os.makedirs(dir_path, exist_ok=True)
print("✓ Directory structure created")

# --- 5. FUNCTION TO COLLECT, SPLIT, AND COPY FILES ---
def process_and_copy_files(source_paths, dest_train, dest_val, val_split=0.15, file_limit=None):
    """Enhanced file processing with better validation split"""
    all_files = []
    for path in source_paths:
        if not os.path.exists(path): 
            continue
        all_files.extend(glob.glob(os.path.join(path, '**', '*.jpg'), recursive=True))
        all_files.extend(glob.glob(os.path.join(path, '**', '*.png'), recursive=True))
        all_files.extend(glob.glob(os.path.join(path, '**', '*.jpeg'), recursive=True))
    
    if file_limit and len(all_files) > file_limit:
        all_files = random.sample(all_files, file_limit)
    
    # Better split - 85/15 for more training data
    train_files, val_files = train_test_split(
        all_files, 
        test_size=val_split, 
        random_state=42, 
        shuffle=True
    )
    
    print(f"  Copying {len(train_files):,} training files...")
    for f in tqdm(train_files, desc="  Train"): 
        shutil.copy(f, dest_train)
    
    print(f"  Copying {len(val_files):,} validation files...")
    for f in tqdm(val_files, desc="  Val"): 
        shutil.copy(f, dest_val)
    
    return len(train_files), len(val_files)

# --- 6. EXECUTE THE PROCESS ---
FILE_LIMIT_PER_CLASS = 50000  # Increased from 40k for more data

print("\n" + "=" * 70)
print(" PROCESSING REAL FACES ".center(70, "="))
print("=" * 70)
real_train, real_val = process_and_copy_files(
    real_face_paths, train_real_dir, val_real_dir, 
    file_limit=FILE_LIMIT_PER_CLASS
)

print("\n" + "=" * 70)
print(" PROCESSING AI-GENERATED FACES ".center(70, "="))
print("=" * 70)
ai_train, ai_val = process_and_copy_files(
    ai_face_paths, train_ai_dir, val_ai_dir, 
    file_limit=FILE_LIMIT_PER_CLASS
)

# --- 7. FINAL VERIFICATION ---
print("\n" + "=" * 70)
print(" DATASET SUMMARY ".center(70, "="))
print("=" * 70)
print(f"Training Set:")
print(f"  Real Images:        {len(os.listdir(train_real_dir)):,}")
print(f"  AI-Generated:       {len(os.listdir(train_ai_dir)):,}")
print(f"  Total:              {len(os.listdir(train_real_dir)) + len(os.listdir(train_ai_dir)):,}")
print(f"\nValidation Set:")
print(f"  Real Images:        {len(os.listdir(val_real_dir)):,}")
print(f"  AI-Generated:       {len(os.listdir(val_ai_dir)):,}")
print(f"  Total:              {len(os.listdir(val_real_dir)) + len(os.listdir(val_ai_dir)):,}")
print("=" * 70)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. Configuration ---
test_dir = '/kaggle/working/dataset/test'
model_path = 'ai_face_detector_efficientnetb2.pth' # Loading the model you just saved
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 260
BATCH_SIZE = 32

# --- 2. Load the Trained Model ---
model = timm.create_model('efficientnet_b2', pretrained=False, num_classes=1)
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

# --- 3. Create the Test DataLoader ---
test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# --- 4. Get Predictions ---
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating Final Model"):
        images = images.to(device)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds).flatten()
all_labels = np.array(all_labels)

# --- 5. Calculate and Print Final Metrics ---
accuracy = accuracy_score(all_labels, all_preds)
print(f"\nFINAL TEST ACCURACY: {accuracy * 100:.2f}%")

class_names = ['ai_generated', 'real']
print("\nFinal Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Generating Final Confusion Matrix...")
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Final Model')
plt.show()